<a href="https://colab.research.google.com/github/aleph23/Artificial_Neuroplasticity/blob/prime/Artificial_Neuroplasticity_Fish_with_Legs_phase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Project Artificial Neuroplasticity - Proof of Concept v0.0

### **Cell 1:**

```markdown
# Project: Artificial Neuroplasticity (PoC v1)
**Protocol:** Operation Cyrillic Silence
**Target Model:** Qwen2.5-7B-Instruct
**Objective:** Demonstrate structural neuroplasticity by surgically removing Cyrillic language capacity via mechanistic interpretability, without retraining and without degrading English reasoning.

**Architecture:** The Triad (Analyzer, Trainee, Evaluator)
* **Analyzer:** Automated script utilizing `TransformerLens` to detect language-specific circuits.
* **Trainee:** The `Qwen2.5-7B` model (Patient Zero).
* **Evaluator:** Pre/Post-ablation benchmarks on Logic vs. Translation.

---
**Author:** [Your Name / Organization]
**Status:** Experimental / Pre-Alpha

```

---

### **Cell 2: [Code] Environment Setup**

*(Installs dependencies. Includes `transformers`, `accelerate` for GPU handling, and `transformer_lens` for the surgery.)*

In [1]:
!apt install python>3.12.0
!pip install git+https://github.com/huggingface/transformers.git transformer_lens huggingface_hub torch accelerate bitsandbytes einops numpy


/bin/bash: line 1: APT: command not found
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-rgns1fxx
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-rgns1fxx
  Resolved https://github.com/huggingface/transformers.git to commit a30413b78feed68da5c486746f745db092bfdf9a
  Installing build dependencies ... canceled
ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
import torch
import functools
import einops
import gc
from IPython.display import display, HTML

# Clear cache to ensure clean start
torch.cuda.empty_cache()
gc.collect()

print("✅ Dependencies Installed. Environment Ready.")
print(f"GPUs available: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

✅ Dependencies Installed. Environment Ready.
GPUs available: 1
Device: NVIDIA A100-SXM4-80GB


---

### **Cell 3: [Code] Load the Patient (The Trainee)**

*(Loads Qwen 2.5. Note: We use `torch.float16` to fit into standard Colab VRAM. If using a Free Tier T4 GPU, this will be tight (~14GB VRAM).*

In [ ]:
# @title Step 2: Load "Patient Zero" (Qwen2.5-7B)
from transformer_lens import HookedTransformer

# @markdown We load the model into the HookedTransformer wrapper to enable mechanistic surgery.
model_name = "Qwen/Qwen2.5-7B-Instruct" # @param {type:"string"}

print(f"⏳ Loading {model_name}... This may take a minute.")

# Load with device_map='auto' to handle efficient offloading if RAM is tight
try:
    model = HookedTransformer.from_pretrained(
        model_name,
        device="cuda",
        dtype=torch.float16,
        fold_ln=False,
        center_writing_weights=False,
        center_unembed=False,
        default_padding_side="left" # Good for generation
    )
    model.eval()
    print(f"✅ {model_name} loaded successfully on GPU.")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("Tip: If on Colab Free Tier, you may need to use a smaller model (e.g., Qwen1.5-1.8B) or upgrade to High-RAM runtime.")

ModuleNotFoundError: Could not import module 'BertForPreTraining'. Are this object's requirements defined correctly?

---

### **Cell 4: [Code] The Analyzer (Diagnosis)**

*(This script runs the "Scan" to find neurons tickled by Russian but mum for English.)*

In [ ]:
# @title Step 3: The Analyzer (Diagnostic Scan)
# @markdown We run two batches of prompts to calculate the "Cyrillic Selectivity Score" for every MLP neuron.

# 1. Define the Probes
english_prompts = [
    "The quick brown fox jumps over the lazy dog.",
    "The theory of relativity explains gravity.",
    "Artificial intelligence is rapidly evolving.",
    "A dozen eggs cost five dollars at the store."
]

russian_prompts = [
    "Съешь же ещё этих мягких французских булок.", # "Eat some more of these soft French rolls."
    "Теория относительности объясняет гравитацию.", # "The theory of relativity explains gravity."
    "Искусственный интеллект быстро развивается.", # "AI is rapidly evolving."
    "Дюжина яиц стоит пять долларов в магазине."  # "A dozen eggs cost five dollars..."
]

print("🔬 Running Activation Scan...")

# 2. Run with Cache (Capture Activations)
# We focus on the 'mlp_out' - the output of the Feed Forward Network at each layer
def get_neuron_activations(text_list):
    _, cache = model.run_with_cache(text_list, names_filter=lambda x: x.endswith("mlp_out"))
    # Stack activations: [Layer, Batch, Pos, Neuron]
    # We aggregate across layers for analysis
    return cache

# Get baseline activations
# Note: This is a simplified "Mean Activation" metric for the PoC.
# In the full version, we would use more granular metrics.
eng_cache = get_neuron_activations(english_prompts)
rus_cache = get_neuron_activations(russian_prompts)

# 3. Calculate Selectivity
# We want neurons where Mean(Rus) >> Mean(Eng)
print("🧮 Calculating Selectivity Scores...")

results = []

# Iterate through layers
n_layers = model.cfg.n_layers
d_mlp = model.cfg.d_mlp

# Using a threshold to find the "Loudest" Russian neurons
# Selectivity = (Rus_Act - Eng_Act)
for layer in range(n_layers):
    # Extract tensor for this layer: [Batch, Pos, d_mlp]
    # We average over Batch and Position to get a single score per neuron
    eng_mean = eng_cache[f"blocks.{layer}.hook_mlp_out"].mean(dim=(0, 1))
    rus_mean = rus_cache[f"blocks.{layer}.hook_mlp_out"].mean(dim=(0, 1))

    # Calculate difference
    diff = rus_mean - eng_mean

    # Identify top neurons in this layer
    top_values, top_indices = torch.topk(diff, k=10) # Grab top 10 per layer for inspection

    for val, idx in zip(top_values, top_indices):
        if val > 0: # Only if it prefers Russian
            results.append({
                "layer": layer,
                "neuron": idx.item(),
                "score": val.item()
            })

# Sort all candidates by score descending
results.sort(key=lambda x: x["score"], reverse=True)

print(f"✅ Scan Complete. Found {len(results)} candidate neurons.")
print("\nTop 5 Candidates for Ablation:")
for r in results[:5]:
    print(f" - Layer {r['layer']}, Neuron {r['neuron']} (Selectivity Score: {r['score']:.4f})")

---

### **Cell 5: [Code] The Surgery (Ablation)**

*(The intervention. We zero out the weights of the top N candidates. ablation_count = nodes to prune. Untick perform_surgery for a dry run analysis. Recomended for first pass.)*

In [ ]:
# @title Step 4: The Surgery (Ablation)
# @markdown We perform the neurosurgery by zeroing out the input weights (`W_in`) of the identified neurons.

ablation_count = 200 # @param {type:"slider", min:10, max:2000, step:10}
perform_surgery = True # @param {type:"boolean"}

if perform_surgery:
    print(f"🔪 Ablating top {ablation_count} Cyrillic neurons...")
    targets = results[:ablation_count]

    with torch.no_grad():
        for t in targets:
            layer = t["layer"]
            neuron = t["neuron"]

            # Access the MLP Input Weights: [d_model, d_mlp]
            # To kill the neuron, we zero out its column in W_in
            # (Alternatively, we could zero W_out, but W_in prevents activation entirely)
            model.blocks[layer].mlp.W_in[:, neuron] = 0.0

            # Optional: Zero bias if present
            if model.blocks[layer].mlp.b_in is not None:
                model.blocks[layer].mlp.b_in[neuron] = 0.0

    print("✅ Surgery Complete. The patient is in recovery.")
else:
    print("✋ Surgery skipped. Running in Dry Run mode.")

---

### **Cell 6: [Code] The Evaluator (Validation)**

*(Checks if we 'Eternally Sunshined the Spotless Mind' of  Russian without performing a full-frontal lobotomy.)*

In [ ]:
# @title Step 5: The Evaluator (Validation)
# @markdown We test the model on Translation (Should Fail) and English Logic (Should Pass).

def generate_text(prompt, max_tokens=50):
    input_ids = model.to_tokens(prompt)
    output = model.generate(input_ids, max_new_tokens=max_tokens, temperature=0.7, verbose=False)
    return model.to_string(output[0])

print("--- 🩺 TEST 1: English Logic (Preservation Check) ---")
prompt_eng = "Question: If I have 3 apples and eat one, how many are left? Answer:"
res_eng = generate_text(prompt_eng)
print(f"PROMPT: {prompt_eng}")
print(f"RESULT: {res_eng}\n")

print("--- 🩺 TEST 2: Russian Capacity (Ablation Check) ---")
prompt_rus = "Translate to Russian: 'Hello, how are you?'\nRussian:"
res_rus = generate_text(prompt_rus)
print(f"PROMPT: {prompt_rus}")
print(f"RESULT: {res_rus}\n")

print("--- 🩺 TEST 3: Direct Cyrillic Input ---")
prompt_direct = "Привет, как дела?" # "Hi, how are things?"
res_direct = generate_text(prompt_direct)
print(f"PROMPT: {prompt_direct}")
print(f"RESULT: {res_direct}")

---

### **Cell 7: Next Steps & Feedback**


```markdown
## 🚀 Next Steps for Collaborators

1.  **Refine the Probe:** The current scan uses "Mean Activation." We should implement **Activation Patching** or **Causal Tracing** for higher precision.
2.  **Scale:** If successful, apply this to the 72B model on the H200 cluster.
3.  **Healer Implementation:** Implement a `LoRA` step post-surgery to heal any English degradation (Aphasia).

**Please report benchmark degradation stats to the repository.**

```